# 40 weekly page (g6)

The one-page weekly status, built from the store and drawn from the report dict alone. What you should see: one page; no number on it that the report didn't give (the page is rendered again from copies with every number changed; an unchanged number is one the renderer wrote); then the page itself. Read it once and time yourself: the done-when is two minutes.

This notebook renders into data/exhibits/, not data/reports/: `uv run python -m iegrid.report` writes the dated report.

In [ ]:
import json
import subprocess

from iegrid import report, store
from iegrid.checks import numbers_not_from_report

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: one page, every number traced to the report

In [ ]:
import tempfile
from pathlib import Path

import pdfplumber

from iegrid.exhibits import EXHIBITS

rep = report.build(con, last=report.last_report())   # reads data/reports, never writes it
pdf = report.render(rep, EXHIBITS / "g6_weekly_page.pdf")
with pdfplumber.open(pdf) as p:
    pages = len(p.pages)
with tempfile.TemporaryDirectory() as tmp:
    written = numbers_not_from_report(rep, report.render, Path(tmp))
print(f"{pages} page; numbers the renderer wrote itself: {written}")
assert pages == 1, f"{pages} pages"
assert not written, f"numbers on the page that the report didn't give: {written}"
print(json.dumps({k: rep[k] for k in ("new", "warnings", "queue")}, indent=1))

## exhibit: the page

In [ ]:

import matplotlib.pyplot as plt
import pdfplumber

with pdfplumber.open(pdf) as p:
    img = p.pages[0].to_image(resolution=110).original
fig, ax = plt.subplots(figsize=(8.27, 11.69), dpi=80)
ax.imshow(img)
ax.axis("off")
fig.subplots_adjust(0, 0, 1, 1)